In [7]:
import pandas as pd
import numpy as np
import pickle
import difflib
from pathlib import Path
from scipy import sparse
from sklearn.neighbors import NearestNeighbors

# ============================================
# Load artifacts
# ============================================
DATA_DIR = Path("../../Data/New")

df = pd.read_csv(DATA_DIR / "Dataset_Final.csv")

with open(DATA_DIR / "tfidf_vectorizer.pkl", "rb") as f:
    tfidf = pickle.load(f)

tfidf_matrix = sparse.load_npz(DATA_DIR / "tfidf_matrix.npz")

assert df.shape[0] == tfidf_matrix.shape[0], "Row count mismatch between dataset and TF-IDF matrix!"
print("Dataset shape:", df.shape)
print("TF-IDF matrix shape:", tfidf_matrix.shape)

Dataset shape: (367703, 7)
TF-IDF matrix shape: (367703, 50000)


In [8]:
# ============================================
# Train and save NearestNeighbors model
# ============================================
nn_model = NearestNeighbors(
    metric='cosine',
    algorithm='brute',
    n_neighbors=50
)
nn_model.fit(tfidf_matrix)

with open(DATA_DIR / "nearest_neighbors.pkl", "wb") as f:
    pickle.dump(nn_model, f)

print("NearestNeighbors model trained and saved.")

NearestNeighbors model trained and saved.


In [9]:
# ============================================
# Title -> row index lookup (dedup, keep first)
# ============================================
title_to_index = pd.Series(df.index, index=df['title'])
title_to_index = title_to_index[~title_to_index.index.duplicated(keep='first')]

def recommend(movie_title, top_n=10):
    """
    Returns top_n movies similar to `movie_title`, sorted by similarity_score desc.
    Returns None and prints a message (with close-match suggestions) if not found.
    """
    if movie_title not in title_to_index:
        close = difflib.get_close_matches(movie_title, title_to_index.index, n=5, cutoff=0.6)
        msg = f"'{movie_title}' not found in dataset."
        if close:
            msg += f" Did you mean: {close}?"
        print(msg)
        return None

    idx = title_to_index[movie_title]

    # queries the model's default 50 neighbors, including the movie itself
    distances, indices = nn_model.kneighbors(tfidf_matrix[idx])
    distances, indices = distances.flatten(), indices.flatten()

    results = []
    for dist, i in zip(distances, indices):
        if i == idx:
            continue  # exclude the movie itself
        results.append({
            'title': df.iloc[i]['title'],
            'similarity_score': round(1 - dist, 4),
            'vote_average': df.iloc[i]['vote_average'],
            'release_date': df.iloc[i]['release_date'],
            'poster_path': df.iloc[i]['poster_path']
        })

    rec_df = (pd.DataFrame(results)
              .sort_values('similarity_score', ascending=False)
              .head(top_n)
              .reset_index(drop=True))
    return rec_df

# Test
recommend("Interstellar", top_n=10)

,title,similarity_score,vote_average,release_date,poster_path
0,To an Asteroid and Back,0.2679,0.000,2024-09-24,/k0evZx3erCK7ZuUran3lt5KjaFf.jpg
1,The Martian,0.2167,8.000,2015,https://m.media-amazon.com/images/M/MV5BMTc2MT...
2,Emergency! [Pilot],0.1990,8.000,1972-01-15,/c6iHxmdx3we2vHc07PfWXAMGKqx.jpg
3,Above and Beyond: NASA's Journey to Tomorrow,0.1828,7.700,2018-10-13,/b7sdRZJSmNvbKfJLTrMSokYPsft.jpg
4,Inside 'Interstellar',0.1801,6.917,2015-03-31,/xWA7Bbo9VeahPcKCFYiC7NZXNoV.jpg
5,The Last Days of Earth,0.1800,0.000,2017-11-17,/zLMEDn3WWbJjI2zsAF73qBK0ANg.jpg
6,The Flying Painter,0.1776,7.000,1937-05-03,/77BVxVKfxa4qiNmuGQUZRiv6P2W.jpg
7,Brother in Every Inch,0.1747,6.300,2022-03-03,/ea39gWMSDU1YjgmvR0UlpAOuYIC.jpg
8,Space Captain: Captain of Space!,0.1739,10.000,2014-10-04,/8fftz3pCKZ0N83MxxUDjfFF5Jpd.jpg
9,Passengers,0.1696,6.956,2016,/mUgzf27luTXvxrzH0jKgSNPwhMC.jpg


## Dataset 2

In [10]:
import pickle
from pathlib import Path
from scipy import sparse
from sklearn.neighbors import NearestNeighbors

DATA_DIR = Path("../../Data/New")  # adjust if running from a different notebook location

tfidf_matrix2 = sparse.load_npz(DATA_DIR / "tfidf_matrix2.npz")

nn_model2 = NearestNeighbors(
    metric='cosine',
    algorithm='brute',
    n_neighbors=50
)
nn_model2.fit(tfidf_matrix2)

with open(DATA_DIR / "nearest_neighbors2.pkl", "wb") as f:
    pickle.dump(nn_model2, f)

print("nearest_neighbors2.pkl saved.")

nearest_neighbors2.pkl saved.
